In [61]:
import plwordnet

wn = plwordnet.load('plwordnet_4_2/plwordnet_4_2.xml')

print(dir(wn))
print(wn)

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_get_hypernym_relations', '_get_hyponym_relations', 'clean', 'description_count', 'description_errors', 'dump', 'find', 'hypernym_paths', 'hypernyms', 'hyponyms', 'lexical_relations', 'lexical_relations_o', 'lexical_relations_p', 'lexical_relations_s', 'lexical_relations_where', 'lexical_units', 'lexical_units_by_name', 'load', 'parse_descriptions', 'relation_by_name', 'relation_types', 'sentiment_count', 'show_relations', 'synset_relations', 'synset_relations_o', 'synset_relations_p', 'synset_relations_s', 'synset_relations_where', 'synsets']
Słowosieć
  lexical units: 513410
  synsets: 353585
  relation types: 306
  synset relat

In [127]:
def synonyms_and_related(word):
    results = set()
    lus = wn.find(word)
    if lus is None:
        return results  # brak wyników → zwracamy pusty set
    for lu in lus:
        synset = lu.synset
        # synonimy w synsecie
        for other in synset.lexical_units:
            results.add(other.name)
        # relacje leksykalne typu "synonim", "przypomina"
        for s, p, o in wn.lexical_relations_where(subject=lu):
            if "synonim" in str(p) or "przypomina" in str(p):
                results.add(o.name)
    return results





In [98]:
print(synonyms_and_related("Konfrontacja"))


['bój', 'czyn zbrojny', 'konflikt', 'konflikt zbrojny', 'konfrontacja', 'porównanie', 'starcie', 'szczęk oręża', 'walka', 'zmagania']


In [ ]:

for lu in wn.find('zawiadomienie'):
    for s, p, o in wn.lexical_relations_where(subject=lu):
        print(p.format(s, o))

In [128]:
import string
import simplemma

def lemmatize(word):
    return simplemma.lemmatize(word, lang="pl")

# funkcja do pobierania synonimów i relacji
def synonyms_and_related_extended(word):
    results = set()
    for lu in wn.find(word):
        synset = lu.synset
        # synonimy w synsecie
        for other in synset.lexical_units:
            results.add(other.name)
        # relacje leksykalne typu "synonim", "przypomina"
        for s, p, o in wn.lexical_relations_where(subject=lu):
            if "synonim" in str(p) or "przypomina" in str(p):
                results.add(o.name)
    return results

def group_synonyms_advanced(text):
    words = [w.strip(string.punctuation) for w in text.split() if w.strip(string.punctuation)]
    words_lem = [lemmatize(w) for w in words]
    words_in_text = set(words_lem)
    
    grouped = {}
    seen = set()
    
    for w in words_lem:
        if w in seen:
            continue
        
        syns = synonyms_and_related_extended(w)
        syns.add(w)
        syns_in_text = syns & words_in_text
        
        if len(syns_in_text) < 2:  # pomijamy pojedyncze słowa
            continue
        
        # łączenie grup jeśli zachodzą wspólne słowa
        found_group = False
        for rep, group in grouped.items():
            if rep in syns_in_text or any(s in syns_in_text for s in group):
                grouped[rep].update(syns_in_text)
                found_group = True
                break
        
        if not found_group:
            grouped[w] = set(syns_in_text)
        
        seen.update(syns_in_text)
    
    # konwersja setów na listy
    for k in grouped:
        grouped[k] = sorted(grouped[k])
    
    return grouped



In [130]:
# tekst = "Leśny krajobraz wokół jeziora zachwyca każdego turystę. Dziki las rośnie gęsto, tworząc malownicze zakątki. Piękny zachód słońca odbija się w tafli wody. Okolica pełna jest kolorowych kwiatów i zielonych drzew. Malowniczy pejzaż przyciąga fotografów z całego świata. Szlachetny spokój tego miejsca koi zmęczone dusze. Wysokie drzewa dają cień i chronią przed wiatrem. Cudowny widok z góry zapiera dech w piersiach. Dzieci bawiące się w lesie cieszą się naturą. Dzikość przyrody sprawia, że każdy dzień jest wyjątkowy."

with open("tekst.txt", "r", encoding="utf-8") as f:
    tekst = f.read()

wynik = group_synonyms_advanced(tekst)
for rep, group in wynik.items():
    print(f"{rep}: {group}")

TypeError: 'NoneType' object is not iterable

In [133]:
import re

def lemmatize(word):
    return simplemma.lemmatize(word, lang="pl")

# pobieranie synonimów
def synonyms_and_related_extended(word):
    results = set()
    lus = wn.find(word)
    if lus is None:
        return results
    for lu in lus:
        synset = lu.synset
        for other in synset.lexical_units:
            results.add(other.name)
        for s, p, o in wn.lexical_relations_where(subject=lu):
            if "synonim" in str(p) or "przypomina" in str(p):
                results.add(o.name)
    return results

# grupowanie słów w synonimy
def group_synonyms_from_text(text):
    # tokenizacja: tylko słowa (litery a-z, ąćęłńóśźż)
    words = re.findall(r'\b[a-zA-ZąćęłńóśźżĄĆĘŁŃÓŚŹŻ]+\b', text)
    words_lem = [lemmatize(w) for w in words]
    words_in_text = set(words_lem)
    
    grouped = {}
    seen = set()
    
    for w in words_lem:
        if w in seen:
            continue
        
        syns = synonyms_and_related_extended(w)
        syns.add(w)
        syns_in_text = syns & words_in_text
        
        if len(syns_in_text) < 2:  # pomijamy pojedyncze słowa
            continue
        
        found_group = False
        for rep, group in grouped.items():
            if rep in syns_in_text or any(s in syns_in_text for s in group):
                grouped[rep].update(syns_in_text)
                found_group = True
                break
        
        if not found_group:
            grouped[w] = set(syns_in_text)
        
        seen.update(syns_in_text)
    
    # konwersja setów na listy
    for k in grouped:
        grouped[k] = sorted(grouped[k])
    
    return grouped

# --- Wczytanie tekstu z pliku ---
with open("tekst.txt", "r", encoding="utf-8") as f:
    tekst = f.read()

# --- Grupowanie słów ---
wynik = group_synonyms_from_text(tekst)

# --- Wyświetlenie wyników ---
for rep, group in wynik.items():
    print(f"{rep}: {group}")

powództwo: ['powództwo', 'pozew']
odpowiedzialność: ['odpowiedzialność', 'odpowiedzialny']
J: ['J', 'j']
o: ['O', 'o']
sąd: ['sąd', 'sądowy', 'zdanie', 'zdać']
powód: ['powodowy', 'powód', 'przyczyna']
kwota: ['kwota', 'suma']
złoty: ['zł', 'złoty']
procesowy: ['proces', 'procesowy']
postępować: ['postępowanie', 'postępować']
właściwy: ['odpowiedni', 'stosowny', 'właściwy']
do: ['C', 'c', 'do']
żądać: ['roszczenie', 'żądanie', 'żądać']
gwarancja: ['gwarancja', 'gwarancyjny', 'rękojmia']
należyty: ['należytość', 'należyty']
wykonanie: ['spełnienie', 'spełnić', 'wykonanie', 'wykonać']
usunięcie: ['usunięcie', 'usunąć']
zobowiązać: ['zobligować', 'zobowiązanie', 'zobowiązać']
pierwszy: ['pierwszy', 'wypłata']
okres: ['chwila', 'moment', 'okres']
ważność: ['dowodzenie', 'istotność', 'treść', 'ważność', 'ważny', 'znaczenie', 'znaczeniowy', 'świadczenie', 'świadczyć']
budowlany: ['budowla', 'budowlany']
wydać: ['wydanie', 'wydać']
uwzględnić: ['uwzględnienie', 'uwzględnić', 'zważyć']
złożyć: